# Pothole Model Workflow (Google Colab Web)

This notebook supports **two modes**:
1. **Train** a new YOLOv8-seg model on your custom dataset.
2. **Import** an already-trained `.pt` model and export it with backend-compatible naming.

Expected backend model file name: `yolov8x-seg-pothole.pt`

## 0) Run in Colab Web
- Open this notebook in **Google Colab (web)**.
- Set runtime: **Runtime → Change runtime type → GPU**.
- Run cells from top to bottom.

In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('Google Drive mounted.')
except Exception as e:
    IN_COLAB = False
    print('This cell must run inside Google Colab web. Error:', e)

This cell must run inside Google Colab web. Error: No module named 'google.colab'


In [ ]:
!pip -q install ultralytics==8.3.0 pyyaml kagglehub[pandas-datasets]

## 1) Choose mode and dataset source
Set `MODE` in the next cell:
- `MODE = "train"` → train a new model from dataset.
- `MODE = "import"` → skip training, import existing trained `.pt`.

For training, dataset source options:
- Kaggle: `srajanchourasia/pothole-dataset`
- Drive zip/folder in YOLO format.

YOLO-seg expected structure:
```
dataset/
  images/train, images/val
  labels/train, labels/val
  data.yaml
```

In [ ]:
from pathlib import Path
import zipfile
import shutil
import yaml

# ====== MAIN SWITCH ======
MODE = 'train'  # 'train' or 'import'

# ====== PATHS ======
DRIVE_ROOT = Path('/content/drive/MyDrive')
WORK_ROOT = Path('/content/work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_DATASET_DIR = WORK_ROOT / 'dataset'

# ====== IMPORT MODE ======
# Primary path for already-trained model
IMPORTED_MODEL_PATH = DRIVE_ROOT / 'ecell-model-exports/yolov8x-seg-pothole.pt'

# Auto-detect fallback if IMPORTED_MODEL_PATH is missing
AUTO_FIND_IMPORTED_MODEL = True
AUTO_FIND_DIRS = [
    DRIVE_ROOT / 'ecell-model-exports',
    DRIVE_ROOT / 'models',
    DRIVE_ROOT,
]
AUTO_FIND_PATTERN = '*.pt'

# If path and auto-find both fail, allow browser upload fallback
ALLOW_UPLOAD_FALLBACK = True

# ====== TRAIN MODE - dataset source ======
USE_KAGGLE_DATASET = True
KAGGLE_DATASET = 'srajanchourasia/pothole-dataset'
KAGGLE_FILE_PATH = ''  # optional pandas preview file path

DATASET_ZIP = DRIVE_ROOT / 'datasets/pothole-seg.zip'
DATASET_DIR_IN_DRIVE = DRIVE_ROOT / 'datasets/pothole-seg'

# ====== TRAIN HYPERPARAMS ======
BASE_MODEL = 'yolov8x-seg.pt'
EPOCHS = 100
IMG_SIZE = 1024
BATCH = 8
PROJECT_NAME = 'pothole-colab-training'
RUN_NAME = 'yolov8x-seg-custom'

# ====== EXPORT ======
EXPORT_DIR = DRIVE_ROOT / 'ecell-model-exports'
EXPORT_FILENAME = 'yolov8x-seg-pothole.pt'

print('Config loaded. MODE =', MODE)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

if MODE == 'train' and USE_KAGGLE_DATASET and KAGGLE_FILE_PATH:
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        KAGGLE_DATASET,
        KAGGLE_FILE_PATH,
    )
    print('First 5 records:')
    display(df.head())
else:
    print('Skipping pandas preview (set MODE="train", USE_KAGGLE_DATASET=True, and KAGGLE_FILE_PATH).')

In [ ]:
if MODE == 'train':
    if LOCAL_DATASET_DIR.exists():
        shutil.rmtree(LOCAL_DATASET_DIR)

    if USE_KAGGLE_DATASET:
        print(f'Using Kaggle dataset: {KAGGLE_DATASET}')
        kaggle_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
        print('Downloaded to:', kaggle_path)
        shutil.copytree(kaggle_path, LOCAL_DATASET_DIR)

        entries = [p for p in LOCAL_DATASET_DIR.iterdir()]
        if len(entries) == 1 and entries[0].is_dir():
            nested = entries[0]
            for item in nested.iterdir():
                shutil.move(str(item), str(LOCAL_DATASET_DIR / item.name))
            nested.rmdir()

    elif DATASET_ZIP and DATASET_ZIP.exists():
        print(f'Using ZIP dataset: {DATASET_ZIP}')
        LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
            zf.extractall(LOCAL_DATASET_DIR)

        entries = [p for p in LOCAL_DATASET_DIR.iterdir()]
        if len(entries) == 1 and entries[0].is_dir():
            nested = entries[0]
            for item in nested.iterdir():
                shutil.move(str(item), str(LOCAL_DATASET_DIR / item.name))
            nested.rmdir()

    elif DATASET_DIR_IN_DRIVE.exists():
        print(f'Using folder dataset: {DATASET_DIR_IN_DRIVE}')
        shutil.copytree(DATASET_DIR_IN_DRIVE, LOCAL_DATASET_DIR)
    else:
        raise FileNotFoundError('Provide Kaggle dataset access or DATASET_ZIP or DATASET_DIR_IN_DRIVE.')

    print('Dataset ready at:', LOCAL_DATASET_DIR)
    print('Top-level:', [p.name for p in LOCAL_DATASET_DIR.iterdir()])

    required = [
        LOCAL_DATASET_DIR / 'images' / 'train',
        LOCAL_DATASET_DIR / 'labels' / 'train',
        LOCAL_DATASET_DIR / 'images' / 'val',
        LOCAL_DATASET_DIR / 'labels' / 'val',
        LOCAL_DATASET_DIR / 'data.yaml',
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise ValueError('Dataset is not in YOLO train/val format. Missing:\n- ' + '\n- '.join(missing))
else:
    print('MODE=import, skipping dataset preparation.')

In [ ]:
if MODE == 'train':
    data_yaml = LOCAL_DATASET_DIR / 'data.yaml'
    if not data_yaml.exists():
        raise FileNotFoundError(f'data.yaml not found at {data_yaml}')

    with open(data_yaml, 'r', encoding='utf-8') as f:
        data_cfg = yaml.safe_load(f)

    data_cfg['path'] = str(LOCAL_DATASET_DIR)

    with open(data_yaml, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data_cfg, f, sort_keys=False)

    print('Updated data.yaml:')
    print(yaml.safe_dump(data_cfg, sort_keys=False))
else:
    print('MODE=import, skipping data.yaml update.')

In [ ]:
from ultralytics import YOLO

if MODE == 'train':
    model = YOLO(BASE_MODEL)
    results = model.train(
        data=str(LOCAL_DATASET_DIR / 'data.yaml'),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH,
        project=str(WORK_ROOT / PROJECT_NAME),
        name=RUN_NAME,
        pretrained=True,
        patience=25,
        workers=2,
        device=0,
    )
    MODEL_TO_EXPORT = Path(results.save_dir) / 'weights' / 'best.pt'
    print('Training complete.')
else:
    MODEL_TO_EXPORT = Path(IMPORTED_MODEL_PATH)

    if not MODEL_TO_EXPORT.exists() and AUTO_FIND_IMPORTED_MODEL:
        print('Configured import path not found. Searching Drive for .pt files...')
        found = []
        for root in AUTO_FIND_DIRS:
            if root.exists():
                found.extend(sorted(root.rglob(AUTO_FIND_PATTERN)))

        if found:
            MODEL_TO_EXPORT = found[0]
            print('Using auto-found model:', MODEL_TO_EXPORT)
            if len(found) > 1:
                print('Other matches:')
                for p in found[1:6]:
                    print('-', p)
        else:
            print('No .pt model found in auto-search directories.')

    if not MODEL_TO_EXPORT.exists() and ALLOW_UPLOAD_FALLBACK:
        print('Upload fallback enabled. Please choose your trained .pt file now.')
        try:
            from google.colab import files
            uploaded = files.upload()
            if uploaded:
                uploaded_name = next(iter(uploaded.keys()))
                MODEL_TO_EXPORT = Path('/content') / uploaded_name
                print('Using uploaded model:', MODEL_TO_EXPORT)
        except Exception as e:
            print('Upload fallback failed:', e)

    if not MODEL_TO_EXPORT.exists():
        raise FileNotFoundError(
            f'Imported model not found. Checked: {IMPORTED_MODEL_PATH}. '
            f'Also searched dirs: {AUTO_FIND_DIRS}. '
            'Set IMPORTED_MODEL_PATH correctly or upload a .pt when prompted.'
        )

    print('Using imported model:', MODEL_TO_EXPORT)

In [ ]:
from ultralytics import YOLO

model_path = Path(MODEL_TO_EXPORT)
print('Selected model:', model_path, 'exists=', model_path.exists())

best_model = YOLO(str(model_path))
if MODE == 'train':
    metrics = best_model.val()
    print(metrics)
else:
    print('MODE=import, validation skipped (run optional predict cell to test).')

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_path = EXPORT_DIR / EXPORT_FILENAME
shutil.copy2(MODEL_TO_EXPORT, export_path)
print('Exported model to:', export_path)

## 2) Import into this project (local machine)
After export, download this file from Drive:
- `MyDrive/ecell-model-exports/yolov8x-seg-pothole.pt`

Place it in this repo at:
- `backend/models/yolov8x-seg-pothole.pt`

Then restart services:
```bash
docker compose up -d --build api celery_worker_inference celery_worker_drone
```

If import mode says model not found:
1. In config cell, set `IMPORTED_MODEL_PATH` to the exact Drive path of your `.pt` file.
2. Keep `AUTO_FIND_IMPORTED_MODEL=True` to auto-search Drive.
3. Keep `ALLOW_UPLOAD_FALLBACK=True` to upload the model directly from browser when prompted.

Quick verification:
1. Upload a drone video in admin dashboard.
2. Confirm inference tasks run in pipeline monitor.
3. Check detections appear on map/list.

In [ ]:
# Optional sanity test (works for both MODE=train and MODE=import)
# sample_img = '/content/drive/MyDrive/path/to/test.jpg'
# pred = best_model.predict(sample_img, conf=0.15, imgsz=IMG_SIZE, save=True)
# print(pred[0].boxes)